In [2]:
import whisper
import datetime
import os
import torch

def format_time(seconds):
    return str(datetime.timedelta(seconds=seconds)).replace(".", ",")[:11]

def create_srt_and_text(segments):
    srt_content = ""
    simple_text = ""  # シンプルなテキスト用の変数
    speaker_text = ""  # 話者情報付きテキスト用の変数
    
    for i, segment in enumerate(segments, start=1):
        start_time = format_time(segment['start'])
        end_time = format_time(segment['end'])
        text = segment['text'].strip()
        
        # SRT形式のコンテンツを生成
        srt_content += f"{i}\n{start_time} --> {end_time}\n{text}\n\n"
        
        # シンプルなテキスト形式（タイムスタンプなし）
        simple_text += f"{text}\n"
        
        # タイムスタンプ付きのテキスト形式
        speaker_text += f"[{start_time}] {text}\n"
    
    return srt_content, simple_text, speaker_text

def ensure_directory_exists(directory):
    if not os.path.exists(directory):
        os.makedirs(directory)

if __name__ == "__main__":
    # ディレクトリ設定
    source_dir = "./source"
    ensure_directory_exists(source_dir)
    
    # 入力ファイルのパス
    input_file = os.path.join(source_dir, "Hirono01.mp3")
    
    # 言語設定
    # 日本語: "ja"
    # 英語: "en"
    # フランス語: "fr"
    # 北京語: "zh"
    LANGUAGE = "ja"

    # whisperのモデルを設定
    print("Whisperモデルをロード中...")
    model = whisper.load_model("large")
    
    try:
        print(f"音声ファイル'{input_file}'を処理中...")
        device = "cuda" if torch.cuda.is_available() else "cpu"
        model.to(device)
        
        result = model.transcribe(input_file, language=LANGUAGE, fp16=False)
        
        base_name = os.path.splitext(os.path.basename(input_file))[0]
        srt_content, simple_text, speaker_text = create_srt_and_text(result["segments"])
        
        # 各形式のファイル名を設定
        srt_file = os.path.join(source_dir, f"{base_name}.srt")
        simple_text_file = os.path.join(source_dir, f"{base_name}_simple.txt")
        speaker_text_file = os.path.join(source_dir, f"{base_name}_with_timestamp.txt")
        
        # ファイルに書き込み
        with open(srt_file, "w", encoding="utf-8") as f:
            f.write(srt_content)
        with open(simple_text_file, "w", encoding="utf-8") as f:
            f.write(simple_text)
        with open(speaker_text_file, "w", encoding="utf-8") as f:
            f.write(speaker_text)
        
        print(f"処理完了:")
        print(f"- SRTファイル: '{srt_file}'")
        print(f"- シンプルテキスト: '{simple_text_file}'")
        print(f"- タイムスタンプ付きテキスト: '{speaker_text_file}'")
        
    except FileNotFoundError:
        print(f"エラー: 入力ファイル '{input_file}' が見つかりません。")
    except Exception as e:
        print(f"エラーが発生しました: {str(e)}")

Whisperモデルをロード中...


C:\Users\Sohei\anaconda3\envs\whisper_drama\lib\site-packages\whisper\__init__.py:146: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(fp, map_location

音声ファイル'./source\Hirono01.mp3'を処理中...
処理完了:
- SRTファイル: './source\Hirono01.srt'
- シンプルテキスト: './source\Hirono01_simple.txt'
- タイムスタンプ付きテキスト: './source\Hirono01_with_timestamp.txt'
